In [32]:
import pandas as pd
df = pd.read_csv("../data/SPY/SPY.USUSD_Candlestick_5_M_ASK_05.10.2022-05.10.2024.csv")
df=df[0:300]

df['Gmt time']=df["Gmt time"].str.replace(".000","")
df['Gmt time'] = pd.to_datetime(df['Gmt time'],format='%d.%m.%Y %H:%M:%S')

df2 = df[['Gmt time','Close','Open']]
df2 = df2.resample('d', on='Gmt time').mean().dropna(how='all')

df2

,Close,Open
Gmt time,,
2022-10-05,377.023837,377.010955
2022-10-06,376.994000,376.994000


In [439]:
import pandas as pd

df = pd.read_csv("../data/SPY/SPY.USUSD_Candlestick_5_M_ASK_05.10.2022-05.10.2024.csv")
#df=df[0:20000]
zoneWidth = .50

df = df[df.notnull().all(axis=1)]
df=df[(df.Volume!= 0)]
df=df[df.High!=df.Low]

df['Gmt time']=df["Gmt time"].str.replace(".000","")
df['datetime']=pd.to_datetime(df['Gmt time'],format='%d.%m.%Y %H:%M:%S')
df['YMD'] = df['datetime'].dt.strftime('%Y%m%d')

df['datetime_est']=pd.to_datetime(df["datetime"], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern')

fromTodayStart = '2022-10-01 09:30:00'
toNow   = '2024-10-01 16:00:00'
df = df[df['datetime_est'].between(fromTodayStart, toNow)].copy()

df['isPriceActionAboveEma25'] = False
df['ema25'] = df['Close'].ewm(span=25, adjust=False).mean()

#df['ema25'] = df.groupby(['YMD'])['Close'].apply(lambda x: x.ewm(span=25, adjust=False).mean()).droplevel(0)
df['isPriceActionAboveEma25'] = (df['Close'] >= df['ema25'])

#cond=abs(df.Close-df.ema25)<=zoneWidth
#df.loc[cond, 'isPriceActionAboveEma25'] = True

df['isTheDayAbove25Ema'] = df.groupby('YMD').isPriceActionAboveEma25.transform(
    lambda x: False if x[x==False].value_counts().shape[0] > 0 else True)

#result = df[df["YMD"]=="20230703"]
result = df[df["isTheDayAbove25Ema"]==True]
result

#result = df[df["YMD"]=="20221028"]
#df.to_csv('examples.csv', index=False)


,Gmt time,Open,High,Low,Close,Volume,datetime,YMD,datetime_est,isPriceActionAboveEma25,ema25,isTheDayAbove25Ema
6786,28.10.2022 13:30:00,379.893,382.244,379.724,381.633,41.499000,2022-10-28 13:30:00,20221028,2022-10-28 09:30:00-04:00,True,380.664093,True
6787,28.10.2022 13:35:00,381.623,382.413,381.513,382.274,38.358600,2022-10-28 13:35:00,20221028,2022-10-28 09:35:00-04:00,True,380.787932,True
6788,28.10.2022 13:40:00,382.263,382.994,382.253,382.553,35.649600,2022-10-28 13:40:00,20221028,2022-10-28 09:40:00-04:00,True,380.923706,True
6789,28.10.2022 13:45:00,382.543,382.974,382.193,382.763,33.381000,2022-10-28 13:45:00,20221028,2022-10-28 09:45:00-04:00,True,381.065190,True
6790,28.10.2022 13:50:00,382.733,382.984,382.224,382.824,34.379700,2022-10-28 13:50:00,20221028,2022-10-28 09:50:00-04:00,True,381.200484,True
...,...,...,...,...,...,...,...,...,...,...,...,...
116008,12.09.2024 19:35:00,558.744,559.084,558.713,558.993,0.000011,2024-09-12 19:35:00,20240912,2024-09-12 15:35:00-04:00,True,557.685240,True
116009,12.09.2024 19:40:00,558.984,559.234,558.793,559.074,0.000012,2024-09-12 19:40:00,20240912,2024-09-12 15:40:00-04:00,True,557.792068,True
116010,12.09.2024 19:45:00,559.083,559.203,558.253,558.443,0.000015,2024-09-12 19:45:00,20240912,2024-09-12 15:45:00-04:00,True,557.842140,True
116011,12.09.2024 19:50:00,558.454,558.804,558.133,558.334,0.000021,2024-09-12 19:50:00,20240912,2024-09-12 15:50:00-04:00,True,557.879975,True


In [ ]:
    df['isPriceActionAboveEma25'] = np.where(df.ema25_5min > df.close_smooth, 
                                             np.where((df.ema25_5min-df.close_smooth)<=zoneWidth, True, False), 
                                            False)